# **Tokenization Algorithm from the ground-up**

Approach: BPE (byte-pair-encoding)

1. Start: every character is its own token
2. Count all adjacent pairs across the corpus
3. Merge the most frequent pair into a new token
4. Repeat steps 2–3 until vocab size is reached (e.g. 50,000)
5. Save the ordered merge rules — that IS the tokenizer

In [1]:
print()

# **Step1: Applying Pre-Tokenization**

In [2]:
def pretokenize(text):

    tokens = []
    current = ""

    for c in text:

        if c == " ":
            if current != "":
                tokens.append(current)
                current = ""
            tokens.append(" ")

        elif c in ",.!?;:":
            if current != "":
                tokens.append(current)
                current = ""
            tokens.append(c)

        else:
            current += c

    if current != "":
        tokens.append(current)

    return tokens

In [3]:
text = "Hello, world! I can't believe it's working so well."
pretokenize(text)

['Hello',
 ',',
 ' ',
 'world',
 '!',
 ' ',
 'I',
 ' ',
 "can't",
 ' ',
 'believe',
 ' ',
 "it's",
 ' ',
 'working',
 ' ',
 'so',
 ' ',
 'well',
 '.']

# **Step 2: Represent text as bytes**

Every piece of text in any language can be represented as a sequence of bytes. A byte is just a number from 0 to 255.

In [4]:
text = "HEllo World!"

tokens1 = []

for character in text:
  char_id = ord(character)
  tokens1.append(char_id)

print(tokens1)
print(len(tokens1))
print(len(text))

[72, 69, 108, 108, 111, 32, 87, 111, 114, 108, 100, 33]
12
12


* ord() takes a single character and returns its Unicode code point
* A code point is just the number assigned to that character
* For basic English characters, this number is the same as its ASCII value
* and also the same as its byte value in UTF-8

In [5]:
text2 = "hello hello hello world"

tokens2 = []
for i in text2:
  tokens2.append(ord(i))

# **Step 3: Count how often each adjacent pair appears**
This is the heart of BPE. The algorithm's entire logic is:

Finding the pair of tokens that appears most frequently next to each other. Merge them into one new token. Repeat.



In [6]:
def count_pairs(token):

  pair_counts = {}

  for i in range(len(token)-1):
    pair = (token[i], token[i + 1])

    if pair not in pair_counts:
      pair_counts[pair] = 0
    pair_counts[pair] += 1

  return pair_counts

In [7]:
count = count_pairs(tokens1)
print(count)

{(72, 69): 1, (69, 108): 1, (108, 108): 1, (108, 111): 1, (111, 32): 1, (32, 87): 1, (87, 111): 1, (111, 114): 1, (114, 108): 1, (108, 100): 1, (100, 33): 1}


In [8]:
counts2 = count_pairs(tokens2)
print(counts2)

{(104, 101): 3, (101, 108): 3, (108, 108): 3, (108, 111): 3, (111, 32): 3, (32, 104): 2, (32, 119): 1, (119, 111): 1, (111, 114): 1, (114, 108): 1, (108, 100): 1}


# **Step 4: Find the most frequent pair**

> Finding which pair has the higest count from our dictionary of pairs and counts




In [9]:
def get_most_frequent_pairs(pair_counts):

  best_pair = None
  best_count = -1

  for pair, count in pair_counts.items():

    if count > best_count:
      best_pair = pair
      best_count = count

  return best_pair, best_count

In [10]:
countPairs = get_most_frequent_pairs(counts2)
print(countPairs)

((104, 101), 3)


# **Step 5: Merge the most frequent pair.**

> Merging the most frequent adjacent tokens. This means walking through the token list, and\ replacing the pair of the most frequent adjacent tokens with a single new token.That new token needs an ID. Since bytes already occupy 0–255, so new merged tokens start at 256 and go up from there.

In [11]:
def merge_pair(tokens, pair_to_merge, new_token_id):

  merged_token = []

  i = 0

  while i < len(tokens):
    if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair_to_merge:
      merged_token.append(new_token_id)
      i += 2
    else:
      merged_token.append(tokens[i])
      i += 1

  return merged_token

In [12]:
new_id = 256
best_pair = countPairs[0]

merged = merge_pair(tokens2, best_pair, new_id)
print(merged)

[256, 108, 108, 111, 32, 256, 108, 108, 111, 32, 256, 108, 108, 111, 32, 119, 111, 114, 108, 100]


# **Step 6: The training loop**
This is where all four pieces connect. fWe run the merge process repeatedly for however many merges we want. Each iteration does: count → find best → merge → record the rule.

The full BPE algorithm does this repeatedly in a loop:

1. Count all pairs
2. Find the best pair
3. Merge it — assign it the next ID
4. Count all pairs again on the updated token list
5. Find the new best pair
6. Merge it — assign it the next ID
7. Repeat until you've done as many merges as you want



In [13]:
def train_bpe(text, num_merges):

  tokens = []
  for character in text:
    tokens.append(ord(character))

  merged_rules = {}
  next_id = 256

  for i in range(num_merges):

    pair_counts = count_pairs(tokens)

    if not pair_counts:
      print(f"No more pairs to merge at: step {i}")
      break

    best_pair, best_count = get_most_frequent_pairs(pair_counts)

    if best_count < 2:
      print(f"Best pair appears only once, stopping at step {i}")
      break

    tokens = merge_pair(tokens, best_pair, next_id)

    merged_rules[best_pair] = next_id

    print(f"Merge {i + 1} {best_pair} -> {next_id} : count was {best_count}")

    next_id += 1

  return tokens, merged_rules



In [14]:
text2 = "hello hello hello world"
final_tokens, merge_rules = train_bpe(text2, num_merges=10)

print("\nFinal tokens:", final_tokens)
print("Merge rules learned:", merge_rules)

Merge 1 (104, 101) -> 256 : count was 3
Merge 2 (256, 108) -> 257 : count was 3
Merge 3 (257, 108) -> 258 : count was 3
Merge 4 (258, 111) -> 259 : count was 3
Merge 5 (259, 32) -> 260 : count was 3
Merge 6 (260, 260) -> 261 : count was 2
Best pair appears only once, stopping at step 6

Final tokens: [261, 260, 119, 111, 114, 108, 100]
Merge rules learned: {(104, 101): 256, (256, 108): 257, (257, 108): 258, (258, 111): 259, (259, 32): 260, (260, 260): 261}


# **Step 7: Encode new text using learned rules**
Right now our merge rules only exist because it has been trained on that specific text.

encoding: Tokenizing a brand new piece of text using the rules we learned

In [15]:
def encode(text, merge_rules):

  tokens = []
  for i in text:
    tokens.append(ord(i))

  for pair, new_id in merge_rules.items():
    tokens = merge_pair(tokens, pair, new_id)

  return tokens

In [16]:
test_text = "hello world"
encoded = encode(test_text, merge_rules)
print(f"Encoded: {encoded}")

test_text2 = "hello hello"
encoded2 = encode(test_text2, merge_rules)
print("Encoded:", encoded2)

test_text3 = "bye world"
encoded3 = encode(test_text3, merge_rules)
print("Encoded:", encoded3)

Encoded: [260, 119, 111, 114, 108, 100]
Encoded: [260, 259]
Encoded: [98, 121, 101, 32, 119, 111, 114, 108, 100]


# **Step 8: Decode tokens back to text**

Decoding: Taking a list of token IDs and reconstructing the original text.

How: Building a vocabulary; a mapping from every token ID to the bytes it represents.

In [17]:
def build_vocab(merge_rules):

  vocab = {}
  for i in range(256):
    vocab[i] = bytes([i])

  for (first, second), new_id in merge_rules.items():
    vocab[new_id] = vocab[first] + vocab[second]

  return vocab

def decode(token_ids, vocab):

  all_bytes = []

  for token_id in token_ids:
    for byte in vocab[token_id]:
      all_bytes.append(byte)

  text = ""
  for byte in all_bytes:
    text += chr(byte)

  return text

In [18]:
vocab = build_vocab(merge_rules)

print(decode([260, 119, 111, 114, 108, 100], vocab))  # should be "hello world"
print(decode([260, 259], vocab))                        # should be "hello hello"
print(decode([98, 121, 101, 32, 119, 111, 114, 108, 100], vocab))  # should be "bye world"

hello world
hello hello
bye world


# **Step 9: Train on a real corpus**


In [19]:
corpus = """
Natural language processing is a subfield of linguistics, computer science, and
artificial intelligence concerned with the interactions between computers and human
language, in particular how to program computers to process and analyze large amounts
of natural language data. The goal is a computer capable of understanding the contents
of documents, including the contextual nuances of the language within them. The
technology can then accurately extract information and insights contained in the
documents, as well as categorize and organize the documents themselves. Challenges in
natural language processing frequently involve speech recognition, natural language
understanding, and natural language generation. Natural language processing has a long
history. The field has evolved significantly with the introduction of deep learning
algorithms which have improved the ability of computers to process and understand
natural language significantly.
"""

In [20]:
class Tokenizer:

    def __init__(self):
        self.merge_rules = {}
        self.vocab = {}
        self.next_id = 256

    # -------------------------
    # PRETOKENIZATION (NOW USED IN TRAIN + ENCODE)
    # -------------------------
    def pretokenize(self, text):

        tokens = []
        current = ""

        for c in text:

            if c == " ":
                if current != "":
                    tokens.append(current)
                    current = ""
                tokens.append(" ")

            elif c in ",.!?;:":
                if current != "":
                    tokens.append(current)
                    current = ""
                tokens.append(c)

            else:
                current += c

        if current != "":
            tokens.append(current)

        return tokens

    # -------------------------
    # CONVERT CHUNK → BYTES
    # -------------------------
    def _get_bytes(self, text):
        tokens = []
        for character in text:
            tokens.append(ord(character))
        return tokens

    # -------------------------
    # COUNT PAIRS
    # -------------------------
    def _count_pairs(self, tokens):

        pair_counts = {}

        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])

            if pair not in pair_counts:
                pair_counts[pair] = 0

            pair_counts[pair] += 1

        return pair_counts

    # -------------------------
    # BEST PAIR
    # -------------------------
    def _get_most_frequent_pair(self, pair_counts):

        best_pair = None
        best_count = -1

        for pair, count in pair_counts.items():
            if count > best_count:
                best_pair = pair
                best_count = count

        return best_pair, best_count

    # -------------------------
    # MERGE
    # -------------------------
    def _merge_pair(self, tokens, pair_to_merge, new_token_id):

        merged_tokens = []
        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair_to_merge:
                merged_tokens.append(new_token_id)
                i += 2
            else:
                merged_tokens.append(tokens[i])
                i += 1

        return merged_tokens

    # -------------------------
    # BUILD VOCAB
    # -------------------------
    def _build_vocab(self):

        vocab = {}

        for i in range(256):
            vocab[i] = bytes([i])

        for (first, second), new_id in self.merge_rules.items():
            vocab[new_id] = vocab[first] + vocab[second]

        self.vocab = vocab

    # -------------------------
    # TRAIN (FIXED: USE PRETOKENIZATION)
    # -------------------------
    def train(self, text, num_merges):

        chunks = self.pretokenize(text)

        tokens = []
        for chunk in chunks:
            tokens.extend(self._get_bytes(chunk))

        for i in range(num_merges):

            pair_counts = self._count_pairs(tokens)
            best_pair, best_count = self._get_most_frequent_pair(pair_counts)

            if best_count < 2:
                print(f"No more useful pairs at step {i}, stopping.")
                break

            tokens = self._merge_pair(tokens, best_pair, self.next_id)

            self.merge_rules[best_pair] = self.next_id

            print(f"Merge {i+1}: {best_pair} -> {self.next_id} count= {best_count}")

            self.next_id += 1

        self._build_vocab()

        print("Training complete.")

    # -------------------------
    # ENCODE (FIXED: PRETOKENIZATION USED)
    # -------------------------
    def encode(self, text):

        chunks = self.pretokenize(text)

        tokens = []
        for chunk in chunks:
            tokens.extend(self._get_bytes(chunk))

        for pair, new_id in self.merge_rules.items():
            tokens = self._merge_pair(tokens, pair, new_id)

        return tokens

    # -------------------------
    # DECODE
    # -------------------------
    def decode(self, token_ids):

        all_bytes = []

        for token_id in token_ids:
            for byte in self.vocab[token_id]:
                all_bytes.append(byte)

        text = ""

        for byte in all_bytes:
            text += chr(byte)

        return text

# **Example Usage**

In [21]:
tokenizer = Tokenizer()

print(tokenizer.train(corpus, num_merges=100))

Merge 1: (101, 32) -> 256 (count=27)
Merge 2: (97, 110) -> 257 (count=26)
Merge 3: (105, 110) -> 258 (count=20)
Merge 4: (115, 32) -> 259 (count=19)
Merge 5: (116, 104) -> 260 (count=15)
Merge 6: (100, 32) -> 261 (count=14)
Merge 7: (97, 108) -> 262 (count=13)
Merge 8: (97, 116) -> 263 (count=12)
Merge 9: (101, 114) -> 264 (count=11)
Merge 10: (101, 110) -> 265 (count=11)
Merge 11: (262, 32) -> 266 (count=10)
Merge 12: (103, 117) -> 267 (count=10)
Merge 13: (99, 111) -> 268 (count=10)
Merge 14: (108, 257) -> 269 (count=9)
Merge 15: (269, 267) -> 270 (count=9)
Merge 16: (270, 97) -> 271 (count=9)
Merge 17: (271, 103) -> 272 (count=9)
Merge 18: (117, 114) -> 273 (count=8)
Merge 19: (272, 256) -> 274 (count=8)
Merge 20: (114, 111) -> 275 (count=8)
Merge 21: (257, 261) -> 276 (count=8)
Merge 22: (32, 260) -> 277 (count=8)
Merge 23: (263, 273) -> 278 (count=7)
Merge 24: (278, 266) -> 279 (count=7)
Merge 25: (279, 274) -> 280 (count=7)
Merge 26: (112, 275) -> 281 (count=7)
Merge 27: (99, 101

In [22]:
encoded = tokenizer.encode("language processing is challenging")
print(encoded)
print(tokenizer.decode(encoded))

[274, 315, 345, 339, 262, 108, 265, 103, 283]
language processing is challenging


# **THE END**